In [1]:
%pip install jupysql pandas matplotlib duckdb-engine
import duckdb
import pandas as pd
import build_geonames_db
from pathlib import Path

if "conn" in globals():
    globals()["conn"].close() # Allow script rerunning

%load_ext sql
conn = build_geonames_db.open_or_init_duckdb(rebuild_views=True)
%config SqlMagic.displaylimit = None
%sql conn --alias duckdb

Note: you may need to restart the kernel to use updated packages.


The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Creating names view...
Creating informal parent city view...
Creating full hierarchy view...
Creating simplified geonames view...


displaylimit: Value None will be treated as 0 (no limit)

In [31]:

print(f"DB Size: {Path(build_geonames_db.DUCK_DB_PATH).stat().st_size / 1e9:.2f} GB")
description = conn.execute("""DESCRIBE""").fetchdf()
description = description[['name', 'column_names', 'column_types']]
description.set_index('name', inplace=True)
description.insert(0, 'row_count', float('nan'))


for table_name in description.index:
    if table_name not in ["fullHierarchy"]:
        row_count = conn.execute(f'SELECT COUNT(*) FROM {table_name}').fetchone()[0]
        description.loc[table_name, 'row_count'] = row_count

display(description.style.format({'row_count': '{:_.0f}'}))

feature_codes = conn.execute(
    """
        SELECT feature_class, feature_code, COUNT(*) AS count
        FROM geonames
        GROUP BY feature_class, feature_code
        ORDER BY count DESC
""").fetchdf()
feature_codes['feature_code'] = feature_codes['feature_class'] + "." + feature_codes['feature_code']
feature_codes.drop(columns=['feature_class'], inplace=True)
feature_codes["proportion"] = feature_codes['count'] / feature_codes['count'].sum()
feature_codes = feature_codes.head(20)
try:
    build_geonames_db.download_file("https://download.geonames.org/export/dump/featureCodes_en.txt", Path("dumps/geonames/featureCodes_en.txt"))
    feature_codes = feature_codes.merge(
        pd.read_csv(
            "dumps/geonames/featureCodes_en.txt", 
            sep='\t', header=None, names=["feature_code", "designation", "description"]
    ), on='feature_code', how='left')
    feature_codes = feature_codes[['feature_code', 'designation', 'count', 'proportion']]
except Exception as e:
    print(f"Could not download feature code descriptions: {e}")
display(
    feature_codes.style
    .format({'count': '{:_.0f}', 'proportion': '{:.0%}'})
    .bar(subset=['proportion'], vmin=0, vmax=1)
)

DB Size: 1.88 GB


,row_count,column_names,column_types
name,,,
allNames,32_284_617,['geonameId' 'isolanguage' 'alternateName' 'isPreferredName' 'isShortName' 'isColloquial' 'isHistoric' 'gndUri'],['INTEGER' 'VARCHAR' 'VARCHAR' 'BOOLEAN' 'BOOLEAN' 'BOOLEAN' 'BOOLEAN' 'VARCHAR']
alternateNames,18_769_801,['alternateNameId' 'geonameId' 'isolanguage' 'alternateName' 'isPreferredName' 'isShortName' 'isColloquial' 'isHistoric'],['INTEGER' 'INTEGER' 'VARCHAR' 'VARCHAR' 'BOOLEAN' 'BOOLEAN' 'BOOLEAN' 'BOOLEAN']
countryInfo,252,['ISO' 'ISO3' 'ISO_Numeric' 'fips' 'Country' 'Capital' 'Area' 'Population' 'Continent' 'tld' 'CurrencyCode' 'CurrencyName' 'Phone' 'Postal_Code_Format' 'Postal_Code_Regex' 'Languages' 'geonameid' 'neighbours' 'EquivalentFipsCode'],['VARCHAR' 'VARCHAR' 'INTEGER' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'FLOAT' 'INTEGER' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'INTEGER' 'VARCHAR' 'VARCHAR']
fullHierarchy,nan,['childId' 'parentId' 'level'],['INTEGER' 'INTEGER' 'VARCHAR']
geonames,13_412_879,['geonameId' 'name' 'asciiname' 'alternatenames' 'latitude' 'longitude' 'feature_class' 'feature_code' 'country_code' 'cc2' 'admin1_code' 'admin2_code' 'admin3_code' 'admin4_code' 'admin5_code' 'population' 'elevation' 'dem' 'timezone' 'modification_date'],['INTEGER' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'FLOAT' 'FLOAT' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'BIGINT' 'INTEGER' 'INTEGER' 'VARCHAR' 'DATE']
gnd,65_617,['gndUri' 'geonameId'],['VARCHAR' 'INTEGER']
gndNames,132_370,['gndUri' 'name' 'isPreferred'],['VARCHAR' 'VARCHAR' 'BOOLEAN']
hierarchy,518_269,['parentId' 'childId' 'type'],['INTEGER' 'INTEGER' 'VARCHAR']
informalParentCity,169_815,['childId' 'parentId'],['INTEGER' 'INTEGER']


,feature_code,designation,count,proportion
0,P.PPL,populated place,4_685_832,35%
1,H.STM,stream,990_177,7%
2,T.HLL,hill,502_096,4%
3,T.MT,mountain,422_594,3%
4,S.FRM,farm,359_812,3%
5,H.LK,lake,309_547,2%
6,S.SCH,school,298_042,2%
7,H.STMI,intermittent stream,261_178,2%
8,S.CH,church,259_635,2%
9,S.HTL,hotel,241_608,2%


In [13]:
%%sql
SELECT COUNT(*) FROM geonames WHERE feature_class = 'P'

Running query in 'duckdb'

count_star()
5195947


5 195 947 populated places in geonames against [2 887 930](https://query.wikidata.org/#SELECT%20%28COUNT%28DISTINCT%20%3Fid%29%20AS%20%3Fcount%29%20WHERE%20%7B%0A%20%20%3Fid%20wdt%3AP31%20%2F%20wdt%3AP279%2a%20wd%3AQ123964505.%0A%7D) on wikidata

However there are [709 681](https://query.wikidata.org/#SELECT%20%28COUNT%28DISTINCT%20%3Fid%29%20AS%20%3Fcount%29%20WHERE%20%7B%0A%20%20%3Fid%20wdt%3AP31%20%2F%20wdt%3AP279%2a%20wd%3AQ123964505.%0A%20%20%3Fid%20wdt%3AP1566%20%3FsomeGeonamesId%0A%7D) wikidata populated places that don't link to a geoname.

Among which [13 554](https://query.wikidata.org/#SELECT%20%28COUNT%28DISTINCT%20%3Fid%29%20AS%20%3Fcount%29%20WHERE%20%7B%0A%20%20%3Fid%20wdt%3AP31%20%2F%20wdt%3AP279%2a%20wd%3AQ253019.%0A%20%20%3Fid%20wdt%3AP1566%20%3FsomeGeonamesId%0A%7D) are ["Ortsteil"](https://www.wikidata.org/wiki/Q253019) including [Sudberg](https://www.wikidata.org/wiki/Q2362997) in Wuppertal, which is present in bzk open but not [geonames](https://www.geonames.org/2805753/wuppertal.html) neither gnd.

In [16]:
%%sql
SELECT levenshtein(nfc_normalize(lower(alternateName)), nfc_normalize(lower('Sudberg'))) AS score, *
FROM allNames NATURAL LEFT JOIN simplifiedGeonames
WHERE score <= 2 AND feature_class = 'P'
ORDER BY score
LIMIT 10;

Running query in 'duckdb'

score,geonameId,isolanguage,alternateName,isPreferredName,isShortName,isColloquial,isHistoric,gndUri,name,asciiname,feature_class,feature_code,country_code,admin1_code,admin2_code,admin3_code,admin4_code,admin5_code,parentCityId
1,2942109,None,Budberg,True,None,None,None,None,Budberg,Budberg,P,PPL,DE,07,051,05170,05170032,None,None
1,2942108,None,Budberg,True,None,None,None,None,Budberg,Budberg,P,PPL,DE,07,059,05974,05974052,None,None
1,11587121,None,Surberg,True,None,None,None,None,Surberg,Surberg,P,PPLL,CH,SG,1724,3275,None,None,None
1,2658465,None,Suberg,True,None,None,None,None,Suberg,Suberg,P,PPL,CH,BE,243,303,None,None,None
1,2942108,None,Budberg,None,None,None,None,None,Budberg,Budberg,P,PPL,DE,07,059,05974,05974052,None,None
1,11962460,se,Sundberg,True,None,None,None,None,Salmenkallio,Salmenkallio,P,PPLX,FI,01,011,091,None,None,None
1,2658465,None,Suberg,None,None,None,None,None,Suberg,Suberg,P,PPL,CH,BE,243,303,None,None,None
1,11587121,de,Surberg,None,None,None,None,None,Surberg,Surberg,P,PPLL,CH,SG,1724,3275,None,None,None
1,2824593,None,Surberg,True,None,None,None,None,Surberg,Surberg,P,PPLA4,DE,02,091,09189,09189148,None,None
1,2942109,None,Budberg,None,None,None,None,None,Budberg,Budberg,P,PPL,DE,07,051,05170,05170032,None,None


In [19]:
%%sql
SELECT levenshtein(nfc_normalize(lower(alternateName)), nfc_normalize(lower('Marienfelde'))) AS score, *
FROM allNames NATURAL LEFT JOIN simplifiedGeonames
WHERE score <= 2 AND feature_class = 'P'
ORDER BY score
LIMIT 10;


Running query in 'duckdb'

score,geonameId,isolanguage,alternateName,isPreferredName,isShortName,isColloquial,isHistoric,gndUri,name,asciiname,feature_class,feature_code,country_code,admin1_code,admin2_code,admin3_code,admin4_code,admin5_code,parentCityId
0,3088128,None,Marienfelde,None,None,None,None,None,Prątnik,Pratnik,P,PPL,PL,85,2802,280202,None,None,None
0,3092300,None,Marienfelde,None,None,None,None,None,Marianka,Marianka,P,PPL,PL,85,2804,280407,None,None,None
0,2873589,None,Marienfelde,True,None,None,None,https://d-nb.info/gnd/5203641-8,Marienfelde,Marienfelde,P,PPLX,DE,16,00,11000,11000000,07,2950159
0,2873592,None,Marienfelde,True,None,None,None,None,Marienfelde,Marienfelde,P,PPL,DE,12,00,13075,13075130,None,None
0,2873591,None,Marienfelde,True,None,None,None,None,Marienfelde,Marienfelde,P,PPL,DE,12,00,13075,13075070,None,None
0,2873593,None,Marienfelde,True,None,None,None,None,Marienfelde,Marienfelde,P,PPL,DE,12,00,13071,13071124,None,None
0,2873591,None,Marienfelde,None,None,None,None,None,Marienfelde,Marienfelde,P,PPL,DE,12,00,13075,13075070,None,None
0,2873589,None,Marienfelde,True,None,None,None,None,Marienfelde,Marienfelde,P,PPLX,DE,16,00,11000,11000000,07,2950159
0,2873590,None,Marienfelde,True,None,None,None,None,Marienfelde,Marienfelde,P,PPL,DE,10,00,01058,01058157,None,None
0,3099326,None,Marienfelde,None,None,None,None,None,Glaznoty,Glaznoty,P,PPL,PL,85,2815,281509,None,None,None


In [20]:
%%sql
SELECT levenshtein(nfc_normalize(lower(child.alternateName)), nfc_normalize(lower('Marienfelde'))) + levenshtein(nfc_normalize(lower(parent.alternateName)), nfc_normalize(lower('Berlin'))) AS score, *
FROM fullHierarchy 
    JOIN (allNames NATURAL LEFT JOIN simplifiedGeonames) AS child ON fullHierarchy.childId = child.geonameId
    JOIN (allNames NATURAL LEFT JOIN simplifiedGeonames) AS parent ON fullHierarchy.parentId = parent.geonameId
WHERE score <= 2 AND child.feature_class = 'P'
ORDER BY score
LIMIT 10;

Running query in 'duckdb'

score,childId,parentId,level,geonameId,isolanguage,alternateName,isPreferredName,isShortName,isColloquial,isHistoric,gndUri,name,asciiname,feature_class,feature_code,country_code,admin1_code,admin2_code,admin3_code,admin4_code,admin5_code,parentCityId,geonameId_1,isolanguage_1,alternateName_1,isPreferredName_1,isShortName_1,isColloquial_1,isHistoric_1,gndUri_1,name_1,asciiname_1,feature_class_1,feature_code_1,country_code_1,admin1_code_1,admin2_code_1,admin3_code_1,admin4_code_1,admin5_code_1,parentCityId_1
0,2873589,2950157,ADM1,2873589,None,Marienfelde,True,None,None,None,https://d-nb.info/gnd/5203641-8,Marienfelde,Marienfelde,P,PPLX,DE,16,00,11000,11000000,07,2950159,2950157,de,Berlin,None,True,None,None,None,Land Berlin,Land Berlin,A,ADM1,DE,16,None,None,None,None,None
0,2873589,2950157,ADM1,2873589,None,Marienfelde,True,None,None,None,None,Marienfelde,Marienfelde,P,PPLX,DE,16,00,11000,11000000,07,2950159,2950157,jbo,berlin,None,None,None,None,None,Land Berlin,Land Berlin,A,ADM1,DE,16,None,None,None,None,None
0,2873589,2950157,ADM1,2873589,None,Marienfelde,True,None,None,None,https://d-nb.info/gnd/5203641-8,Marienfelde,Marienfelde,P,PPLX,DE,16,00,11000,11000000,07,2950159,2950157,None,Berlin,False,None,None,None,https://d-nb.info/gnd/4087295-6,Land Berlin,Land Berlin,A,ADM1,DE,16,None,None,None,None,None
0,2873589,2950157,ADM1,2873589,None,Marienfelde,True,None,None,None,https://d-nb.info/gnd/5203641-8,Marienfelde,Marienfelde,P,PPLX,DE,16,00,11000,11000000,07,2950159,2950157,jbo,berlin,None,None,None,None,None,Land Berlin,Land Berlin,A,ADM1,DE,16,None,None,None,None,None
0,2873589,2950157,ADM1,2873589,None,Marienfelde,True,None,None,None,None,Marienfelde,Marienfelde,P,PPLX,DE,16,00,11000,11000000,07,2950159,2950157,None,Berlin,True,None,None,None,https://d-nb.info/gnd/4005728-8,Land Berlin,Land Berlin,A,ADM1,DE,16,None,None,None,None,None
0,2873589,2950157,ADM1,2873589,None,Marienfelde,True,None,None,None,None,Marienfelde,Marienfelde,P,PPLX,DE,16,00,11000,11000000,07,2950159,2950157,de,Berlin,None,True,None,None,None,Land Berlin,Land Berlin,A,ADM1,DE,16,None,None,None,None,None
0,2873589,2950157,ADM1,2873589,None,Marienfelde,True,None,None,None,https://d-nb.info/gnd/5203641-8,Marienfelde,Marienfelde,P,PPLX,DE,16,00,11000,11000000,07,2950159,2950157,None,Berlin,True,True,None,None,None,Land Berlin,Land Berlin,A,ADM1,DE,16,None,None,None,None,None
0,2873589,2950157,ADM1,2873589,None,Marienfelde,True,None,None,None,None,Marienfelde,Marienfelde,P,PPLX,DE,16,00,11000,11000000,07,2950159,2950157,None,Berlin,False,None,None,None,https://d-nb.info/gnd/4087295-6,Land Berlin,Land Berlin,A,ADM1,DE,16,None,None,None,None,None
0,2873589,2950157,ADM1,2873589,None,Marienfelde,True,None,None,None,https://d-nb.info/gnd/5203641-8,Marienfelde,Marienfelde,P,PPLX,DE,16,00,11000,11000000,07,2950159,2950157,None,Berlin,True,None,None,None,https://d-nb.info/gnd/4005728-8,Land Berlin,Land Berlin,A,ADM1,DE,16,None,None,None,None,None
0,2873589,2950157,ADM1,2873589,None,Marienfelde,True,None,None,None,None,Marienfelde,Marienfelde,P,PPLX,DE,16,00,11000,11000000,07,2950159,2950157,None,Berlin,True,True,None,None,None,Land Berlin,Land Berlin,A,ADM1,DE,16,None,None,None,None,None


In [33]:
%%sql
SELECT levenshtein(nfc_normalize(lower(alternateName)), nfc_normalize(lower('N.Y.'))) AS score, *
FROM allNames NATURAL LEFT JOIN simplifiedGeonames
WHERE score <= 2 AND feature_class = 'P'
ORDER BY score
LIMIT 10;

Running query in 'duckdb'

score,geonameId,isolanguage,alternateName,isPreferredName,isShortName,isColloquial,isHistoric,gndUri,name,asciiname,feature_class,feature_code,country_code,admin1_code,admin2_code,admin3_code,admin4_code,admin5_code,parentCityId
2,5164561,ca,Ney,None,None,None,None,None,Ney,Ney,P,PPL,US,OH,039,81200,None,None,None
2,2687824,lld,Nye,None,None,None,None,None,Nye,Nye,P,PPL,SE,08,0685,None,None,None,None
2,9945757,None,Nayi,None,None,None,None,None,Nayi,Nayi,P,PPL,CN,16,4512,None,None,None,None
2,2301174,None,Nkyi,None,None,None,None,None,Enkye,Enkye,P,PPL,GH,18,112,None,None,None,None
2,9895788,None,Nayu,None,None,None,None,None,Nayu,Nayu,P,PPL,CN,16,4501,None,None,None,None
2,9927321,None,Naye,None,None,None,None,None,Naye,Naye,P,PPL,CN,16,4510,None,None,None,None
2,1449275,None,Nay,None,None,None,None,None,Nay,Nay,P,PPL,AF,41,3409,None,None,None,None
2,122218,None,Neyj,None,None,None,None,None,Neyk,Neyk,P,PPL,IR,41,None,None,None,None,None
2,7654115,None,Nayu,True,None,None,None,None,Nayu,Nayu,P,PPL,CN,31,12324209,None,None,None,None
2,1697840,None,Naya,True,None,None,None,None,Naya,Naya,P,PPL,PH,02,48,025015000,None,None,None


There are ADMD entities that are the only reference of admin1_code. For now, ignore.

In [5]:
%%sql
SELECT country.Country, geonames.* FROM geonames 
    JOIN (
        SELECT child.country_code as country_code, ANY_VALUE(child.geonameId) AS chosen_one FROM geonames AS child
        WHERE 
            starts_with(child.feature_code, 'ADMD') AND 
            child.admin1_code IS NOT NULL AND NOT EXISTS(
                SELECT * FROM geonames AS parent 
                WHERE 
                    parent.country_code = child.country_code AND
                    parent.admin1_code = child.admin1_code AND 
                    starts_with(parent.feature_code, 'ADM1')
            )
        GROUP BY child.country_code
    ) ON chosen_one = geonames.geonameId
    JOIN countryInfo AS country ON geonames.country_code = country.ISO


Running query in 'duckdb'

Country,geonameId,name,asciiname,alternatenames,latitude,longitude,feature_class,feature_code,country_code,cc2,admin1_code,admin2_code,admin3_code,admin4_code,admin5_code,population,elevation,dem,timezone,modification_date
Botswana,443222,Bakwena Tribal Territory,Bakwena Tribal Territory,None,-24.583330154418945,25.799999237060547,A,ADMD,BW,None,00,None,None,None,None,0,None,1015,Africa/Gaborone,2000-10-04
Iraq,6687112,Ostān-e Kūt,Ostan-e Kut,"astan kwt,استان کوت",32.899200439453125,46.32870101928711,A,ADMD,IQ,None,00,None,None,None,None,0,None,62,Asia/Baghdad,2008-02-09
Mozambique,1024062,Circunscrição do Zumbo,Circunscricao do Zumbo,None,-15.23723030090332,30.94424057006836,A,ADMD,MZ,None,00,None,None,None,None,0,None,737,Africa/Maputo,2011-07-12
Bulgaria,725796,Voditsa Obshtina,Voditsa Obshtina,"Obshhina Vodica,Община Водица",43.349998474121094,26.049999237060547,A,ADMD,BG,None,00,None,None,None,None,0,None,319,Europe/Sofia,2007-05-30
Philippines,8367019,Bangsamoro Autonomous Region of Muslim Mindanao,Bangsamoro Autonomous Region of Muslim Mindanao,"ARMM,Autonomous Region of Muslim Mindanao,BARMM,Bangsamoro Autonomous Region of Muslim Mindanao",5.976180076599121,121.3868408203125,A,ADMD,PH,None,00,None,None,None,None,0,None,129,Asia/Manila,2020-03-10
Azerbaijan,147557,Martuni Rayonu,Martuni Rayonu,Martuninskiy Rayon,39.75,47.0,A,ADMD,AZ,None,00,None,None,None,None,0,None,930,Asia/Baku,2007-05-30
Georgia,611834,South Ossetia [provisional],South Ossetia [provisional],"Chussar Iryston,Juzhnaja Osetija,Juzna Osetija,Južna Osetija,Khussar Iryston,Samkhret' Oset'is AO,Samkhret' Oset'is Avtonomiuri Olk'i,South Ossetia,South Ossetian Autonomous Oblast,Suedossetien,Südossetien,Yugo-Osetinskaya AO,Yugo-Osetinskaya Avtonoanaya Oblast',samkhreti oseti,Хуссар Ирыстон,Южная Осетия,სამხრეთი ოსეთი",42.33332824707031,44.0,A,ADMD,GE,None,00,None,None,None,None,0,None,1466,Asia/Tbilisi,2022-10-08
Greenland,7422509,Pituffik,Pituffik,Pituffik,76.5445327758789,-68.67862701416016,A,ADMD,GL,None,00,None,None,None,None,0,None,124,America/Thule,2013-12-26
Equatorial Guinea,2306734,Santa Isabel,Santa Isabel,"Administracion Territorial de Santa Isabel,Administración Territorial de Santa Isabel,Demarcacion de Sanda Isabel,Demarcación de Sanda Isabel,Santa Isabel",3.6333301067352295,8.75,A,ADMD,GQ,None,00,None,None,None,None,0,None,1017,Africa/Malabo,2012-01-18
Greece,8200474,Apokentroméni Dioíkisi Attikís,Apokentromeni Dioikisi Attikis,"Apokentromeni Dioikisi Attikis,Apokentroméni Dioíkisi Attikís,Attiki,Attikí,Αποκεντρωμένη Διοίκηση Αττικής,Αττική",38.06932830810547,23.707469940185547,A,ADMD,GR,None,00,None,None,None,None,0,None,159,Europe/Athens,2012-05-06


There are places under multiple cities in the informal hierachy. For now, include all.

In [6]:
%%sql
SELECT 
    childId AS childId,
    child.name AS childName,
    unnest(list(parentId)) AS parentId,
    unnest(list(parent.name)) AS parentName
FROM hierarchy
JOIN geonames AS parent ON hierarchy.parentId = parent.geonameId
JOIN geonames AS child ON hierarchy.childId = child.geonameId
WHERE 
    parent.feature_class = 'P' AND parent.feature_code != 'PPLX' AND
    hierarchy.type IS DISTINCT FROM 'ADM'
GROUP BY childId, child.name
HAVING COUNT(*) > 1;

Running query in 'duckdb'

childId,childName,parentId,parentName
634928,Tapanila,632453,Vantaa
634928,Tapanila,12750226,North Helsinki
643032,Pakila,632453,Vantaa
643032,Pakila,12750226,North Helsinki
798763,Viikki,658225,Helsinki
798763,Viikki,12750226,North Helsinki
641374,Pitäjänmäki,658225,Helsinki
641374,Pitäjänmäki,12750226,North Helsinki
866152,Erebuni,11336814,Khaldi
866152,Erebuni,616052,Yerevan


In [9]:
%%sql
SELECT childId, child.name AS childName, child.feature_code, parentId, parent.name AS parentName, parent.admin1_code, parent.admin2_code, parent.feature_code as parentFeatureCode, level FROM fullHierarchy 
    JOIN geonames AS parent ON fullHierarchy.parentId = parent.geonameId 
    JOIN geonames AS child ON fullHierarchy.childId = child.geonameId
WHERE fullHierarchy.childId = 9408119

Running query in 'duckdb'

childId,childName,feature_code,parentId,parentName,admin1_code,admin2_code,parentFeatureCode,level
9408119,Main Ridge,PPLX,2077456,Commonwealth of Australia,00,None,PCLI,Country
9408119,Main Ridge,PPLX,2145234,State of Victoria,07,None,ADM1,ADM1
9408119,Main Ridge,PPLX,2158177,Melbourne,07,24600,PPLA,City
9408119,Main Ridge,PPLX,2166453,Flinders,07,25340,PPL,City
9408119,Main Ridge,PPLX,7839813,Mornington Peninsula,07,25340,ADM2,ADM2


In [32]:
%%sql
SELECT * FROM fullHierarchy JOIN allNames ON fullHierarchy.parentId = allNames.geonameId 
WHERE fullHierarchy.childId = 9408119 
LIMIT 10

Running query in 'duckdb'

childId,parentId,level,geonameId,isolanguage,alternateName,isPreferredName,isShortName,isColloquial,isHistoric,gndUri
9408119,2077456,Country,2077456,lt,Australija,True,None,None,None,None
9408119,2077456,Country,2077456,ru,Австралия,True,None,None,None,None
9408119,2145234,ADM1,2145234,fy,Victoria,None,None,None,None,None
9408119,2158177,City,2158177,gre,Μελβούρνη,None,None,None,None,None
9408119,2077456,Country,2077456,sco,Australie,None,None,None,None,None
9408119,2158177,City,2158177,it,Melbourne,None,None,None,None,None
9408119,2077456,Country,2077456,sk,Austrália,True,None,None,None,None
9408119,2145234,ADM1,2145234,uz,Viktoriya,None,None,None,None,None
9408119,2145234,ADM1,2145234,sco,Victoria,None,None,None,None,None
9408119,2077456,Country,2077456,tk,Awstraliýa,True,None,None,None,None
